
# Fine-Tuning GPT-2 for Sentiment Classification (IMDB)


### 1. Install and import dependencies

In [1]:
!pip -q install transformers datasets accelerate

In [1]:

import math
import os
import random

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, GPT2ForSequenceClassification, get_linear_schedule_with_warmup

import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("Mixed precision (AMP) enabled by default on CUDA.")
    

PyTorch: 2.10.0+cpu
CUDA available: False


### 2. Configuration & reproducibility

In [2]:

# ---- Experiment config ----
MODEL_NAME = "gpt2"                # smllest GPT-2 version
MAX_LENGTH = 256                   # truncate long reviews for speed
BATCH_SIZE = 8
NUM_EPOCHS = 2                     # keep small for class time
LEARNING_RATE = 5e-5               # conservative for GPT-2 FT
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06                # % of total steps used for LR warmup
GRAD_ACCUM_STEPS = 2               # simulate larger batch on limited GPU

# Subsampling for classroom runs (set to None to use full sets)
SUBSET_TRAIN = 4000
SUBSET_VALID = 2000

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path("gpt2_imdb_runs"); OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Using DEVICE:", DEVICE)
    

Using DEVICE: cpu


### 3. Load the IMDB dataset (binary sentiment)

In this example we will be using the [*IMDB - Large Movie Review Dataset*](https://huggingface.co/datasets/stanfordnlp/imdb). This is a dataset for binary sentiment classification with 25,000 movie reviews for training, and 25,000 for testing. There is additional unlabeled data for use as well.

In [3]:
ds = load_dataset("imdb")
print(ds)

train_raw = ds["train"]
valid_raw = ds["test"]             # use test as validation for demo

# Subsample for faster runs. To use the full dataset, set SUBSET_TRAIN and SUBSET_VALID to None.
if SUBSET_TRAIN is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(SUBSET_TRAIN, len(train_raw))))
if SUBSET_VALID is not None:
    valid_raw = valid_raw.shuffle(seed=SEED).select(range(min(SUBSET_VALID, len(valid_raw))))

print("Train size:", len(train_raw), "Valid size:", len(valid_raw))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train size: 4000 Valid size: 2000


### 4. Tokenization & preprocessing
We use **GPT-2** tokenizer to tokenize the dataset. GPT-2 has **no PAD token** by default, so we set `pad_token = eos_token`. 
We tokenize review texts truncating to `MAX_LENGTH` and keep integer labels (`0` = negative, `1` = positive).


In [4]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Important: set PAD to EOS for GPT-2
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Map labels to integers and back
id2label = {0: "negative", 1: "positive"}
label2id = {v: k for k, v in id2label.items()}


def preprocess_function(batch):
    enc = tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True, padding=False)
    enc["labels"] = batch["label"]
    return enc

# Preprocess training and validation sets by tokenizing, truncating the reviews to MAX_LENGTH and mapping labels to integers. 
# We also remove the original text and label columns since they are no longer needed after tokenization.
train_tokenized = train_raw.map(preprocess_function, batched=True, remove_columns=train_raw.column_names)
valid_tokenized = valid_raw.map(preprocess_function, batched=True, remove_columns=valid_raw.column_names)

print(train_tokenized)
print(valid_tokenized)
    

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})


In [6]:
print(train_tokenized[0])

{'input_ids': [1858, 318, 645, 8695, 379, 477, 1022, 6401, 959, 290, 4415, 5329, 475, 262, 1109, 326, 1111, 389, 1644, 2168, 546, 6590, 6741, 13, 4415, 5329, 3073, 42807, 11, 6401, 959, 3073, 6833, 13, 4415, 5329, 21528, 389, 2407, 2829, 13, 6401, 959, 338, 7110, 389, 1290, 517, 8253, 986, 6401, 959, 3073, 517, 588, 5537, 8932, 806, 11, 611, 356, 423, 284, 4136, 20594, 986, 383, 1388, 2095, 318, 4939, 290, 7650, 78, 11, 475, 423, 366, 27659, 40024, 590, 1911, 4380, 588, 284, 8996, 11, 284, 5052, 11, 284, 13446, 13, 1374, 546, 655, 13226, 30, 40473, 1517, 1165, 11, 661, 3597, 6401, 959, 3073, 1605, 475, 11, 319, 262, 584, 1021, 11, 11810, 484, 4702, 1605, 2168, 357, 10185, 737, 6674, 340, 338, 262, 3303, 11, 393, 262, 4437, 11, 475, 314, 892, 428, 2168, 318, 517, 3594, 621, 1605, 13, 2750, 262, 835, 11, 262, 10544, 389, 1107, 922, 290, 8258, 13, 383, 7205, 318, 407, 31194, 379, 477, 986], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

### 5. Data Loader
We pad to the longest sequence in each batch. Labels remain as integers `0/1`.


In [5]:
def collate_fn(features):
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
    batch = tokenizer.pad(
        {k: [f[k] for f in features] for k in ["input_ids", "attention_mask"]},
        padding=True,
        return_tensors="pt",
    )
    batch["labels"] = labels
    return batch

train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_tokenized, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_loader), len(valid_loader)

(500, 250)

### 6. Model, optimizer, and scheduler

We will be using [`GPT2ForSequenceClassification`](https://huggingface.co/docs/transformers/v5.3.0/en/model_doc/gpt2#transformers.GPT2ForSequenceClassification), a version of GPT adapted with a sequence classification head on top (linear layer). It uses the last token in order to do the classification. 

The specific checkpoint of the model is specified by the parameter `MODEL_NAME`. By default, we will use 'gpt2', the smallest available GPT model with 124M parameters. You can also try larger versions 'gpt2-medium', 'gpt2-large' or 'gpt2-xl' (see https://huggingface.co/openai-community?search_models=gpt)

In [6]:
# We load the pretrained GPT-2 model for sequence classification with 2 output labels (positive/negative). 
model = GPT2ForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# Make sure model knows the pad token
model.config.pad_token_id = tokenizer.pad_token_id
# For safety, if tokenizer was modified (pad set), we can resize embeddings
model.resize_token_embeddings(len(tokenizer))
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
t_total_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * t_total_steps)

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=t_total_steps)

print(f"Total steps: {t_total_steps} | Warmup steps: {num_warmup_steps}")
    

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total steps: 500 | Warmup steps: 30


### 7. Inference with the pre-trained model

We do inference for the first 10 elements in the validation set to check the performanca of the pre-trained model before fine-tuning.

In [ ]:
model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
with torch.no_grad():
    logits = model(**enc).logits
probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
preds = probs.argmax(axis=-1)

for text, p, pr in zip(examples, preds, probs):
    print("Review:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Prediction:", id2label[int(p)], " | P(neg)=", f"{pr[0]:.3f}", " P(pos)=", f"{pr[1]:.3f}")
    print("---")
    


### 8. Training loop


In [7]:
from sklearn.metrics import accuracy_score, f1_score

train_loss_history, valid_loss_history = [], []
valid_acc_history, valid_f1_history = [], []

best_valid_acc = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"Epoch {epoch} [train]")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        # Scales the loss before backpropagation so that gradient accumulation matches the magnitude of a larger effective batch
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        # Only update weights and step scheduler every GRAD_ACCUM_STEPS to simulate larger batch size
        if (step % GRAD_ACCUM_STEPS == 0) or (step == len(train_loader)):
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        # For logging, we accumulate the loss scaled by GRAD_ACCUM_STEPS to reflect the effective batch size.   
        running_loss += loss.item() * GRAD_ACCUM_STEPS
        avg_loss = running_loss / step
        pbar.set_postfix({"loss": f"{avg_loss:.4f}"})

    train_loss_history.append(avg_loss)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch} [valid]")
        for batch in pbar_val:
            labels = batch["labels"].to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            outputs = model(**batch)
            val_loss += outputs.loss.item()
            logits = outputs.logits

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.detach().cpu().tolist())
            all_labels.extend(labels.detach().cpu().tolist())

    mean_val_loss = val_loss / max(1, len(valid_loader))
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    valid_loss_history.append(mean_val_loss)
    valid_acc_history.append(acc)
    valid_f1_history.append(f1)

    np.save(OUT_DIR / "train_loss.npy", np.array(train_loss_history))
    np.save(OUT_DIR / "valid_loss.npy", np.array(valid_loss_history))
    np.save(OUT_DIR / "valid_acc.npy", np.array(valid_acc_history))
    np.save(OUT_DIR / "valid_f1.npy", np.array(valid_f1_history))

    print(f"Epoch {epoch}: train_loss={avg_loss:.4f} | valid_loss={mean_val_loss:.4f} | acc={acc:.3f} | F1={f1:.3f}")

    if acc > best_valid_acc:
        best_valid_acc = acc
        save_dir = "gpt2-imdb-best"
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Saved new best model to: {save_dir}")

Epoch 1 [train]:   0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 9. Plot metrics across epochs
We visualize **training/validation loss** and **validation accuracy/F1**.


In [ ]:

# Reload histories if needed
if 'train_loss_history' not in globals():
    train_loss_history = np.load(OUT_DIR / 'train_loss.npy').tolist()
    valid_loss_history = np.load(OUT_DIR / 'valid_loss.npy').tolist()
    valid_acc_history = np.load(OUT_DIR / 'valid_acc.npy').tolist()
    valid_f1_history = np.load(OUT_DIR / 'valid_f1.npy').tolist()

epochs = list(range(1, len(train_loss_history) + 1))

plt.figure(figsize=(6,4))
plt.plot(epochs, train_loss_history, label='Training loss')
plt.plot(epochs, valid_loss_history, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend(); plt.tight_layout();
plt.savefig(OUT_DIR / 'loss_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs, valid_acc_history, label='Validation Accuracy')
plt.plot(epochs, valid_f1_history, label='Validation F1 (macro)')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Validation Accuracy & F1')
plt.legend(); plt.tight_layout();
plt.savefig(OUT_DIR / 'acc_f1_curves.png', dpi=150)
plt.show()

print('Saved figures to:', OUT_DIR.resolve())
    

### 10. Inference with the fine-tuned model

We do inference for the first 10 elements in the validation set to check the performanca of the fine-tuned model.

In [11]:

model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
with torch.no_grad():
    logits = model(**enc).logits
probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
preds = probs.argmax(axis=-1)

for text, p, pr in zip(examples, preds, probs):
    print("Review:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Prediction:", id2label[int(p)], " | P(neg)=", f"{pr[0]:.3f}", " P(pos)=", f"{pr[1]:.3f}")
    print("---")
    

Review: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it. I truly hope A Thousand Acres has had the same effect on some others out there.<br /><br 

### 11. Trainer variant 

We now replicate the same task using **transformers.Trainer**. This keeps the same tokenizer, collator, and model (GPT2ForSequenceClassification), and uses a `compute_metrics` callback to report **Accuracy** and **Macro-F1**.

In [ ]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    import numpy as np
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    # Macro-F1
    tp = ((preds==1) & (labels==1)).sum()
    tn = ((preds==0) & (labels==0)).sum()
    fp = ((preds==1) & (labels==0)).sum()
    fn = ((preds==0) & (labels==1)).sum()
    def f1_from_counts(tp, fp, fn):
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        return 2*prec*rec/(prec+rec+1e-12)
    f1_pos = f1_from_counts(tp, fp, fn)
    f1_neg = f1_from_counts(tn, fn, fp)
    macro_f1 = float((f1_pos + f1_neg)/2.0)
    acc = float((preds==labels).mean())
    return {'accuracy': acc, 'macro_f1': macro_f1}

trainer_args = TrainingArguments(
    output_dir=str(OUT_DIR / 'trainer_ckpts'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='linear',
    warmup_steps=WARMUP_RATIO,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    fp16=(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    report_to=['none'],
    seed=SEED,
)

trainer_model = GPT2ForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer_model.config.pad_token_id = tokenizer.pad_token_id
trainer_model.resize_token_embeddings(len(tokenizer))

trainer = Trainer(
    model=trainer_model,
    args=trainer_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train and evaluate with Trainer

In [15]:
train_output = trainer.train()
eval_metrics = trainer.evaluate()
print('Eval metrics:', eval_metrics)
trainer.save_model(str(OUT_DIR / 'trainer_final'))


C:\Users\ernest\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

Plot loss curves from Trainer logs

In [ ]:
logs = trainer.state.log_history
tr_ep, tr_loss, ev_ep, ev_loss = [], [], [], []
for r in logs:
    if 'loss' in r and 'epoch' in r:
        tr_ep.append(r['epoch']); tr_loss.append(r['loss'])
    if 'eval_loss' in r and 'epoch' in r:
        ev_ep.append(r['epoch']); ev_loss.append(r['eval_loss'])
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
if tr_ep: plt.plot(tr_ep, tr_loss, label='Training loss (Trainer)')
if ev_ep: plt.plot(ev_ep, ev_loss, label='Validation loss (Trainer)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Trainer loss curves'); plt.legend(); plt.tight_layout(); plt.show()
